In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Refactored Code: Text Classification using Pretrained BERT
# Dataset: AG News (4 classes)
# ─────────────────────────────────────────────────────────────────────────────

# Install transformers if needed
# pip install transformers

import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertTokenizer, TFBertModel
from tensorflow.keras import layers, models, callbacks

In [14]:
train_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/train.csv"
test_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/test.csv"

MODEL_DIR = "/Users/sameerkhan/Desktop/sameerkhan/weights/nlp/bert"
os.makedirs(MODEL_DIR, exist_ok=True)

CHECKPOINT_FILE = f"agnews_bert.h5"

FINAL_MODEL_FILE = "agnews_bert.keras"

VOCAB_FILE = MODEL_DIR + "/agnews_vocab.txt"

GLOVE_FILE = '/Users/sameerkhan/Desktop/sameerkhan/data/nlp/glove.42B.300d.txt'

In [15]:
train_df = pd.read_csv(train_path,header=0)
train_df = train_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})

test_df = pd.read_csv(test_path,header=0)
test_df = test_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})


# zero-based labels
train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"]  = test_df["label"].astype(int) - 1

# combine title + description
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 2) Setup Hyperparameters
# ─────────────────────────────────────────────────────────────────────────────
BERT_MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 1
NUM_CLASSES = 4
AUTOTUNE = tf.data.AUTOTUNE


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 3) Tokenization using BERT Tokenizer
# ─────────────────────────────────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

def encode_texts(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='tf'
    )

train_encodings = encode_texts(train_texts)
test_encodings = encode_texts(test_texts)

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# 4) Create tf.data.Dataset
# ─────────────────────────────────────────────────────────────────────────────
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).shuffle(len(train_texts)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# 5) Build Model with BERT Encoder
# ─────────────────────────────────────────────────────────────────────────────
bert_model = TFBertModel.from_pretrained(BERT_MODEL_NAME)

# Freeze BERT layers (optional for faster training)
bert_model.trainable = False

# Input layers
input_ids = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

# BERT output
bert_outputs = bert_model(
    input_ids,
    attention_mask=attention_mask
)

# Take [CLS] token output (pooled output)
pooled_output = bert_outputs.pooler_output

# Classification head
x = layers.Dense(128, activation='relu')(pooled_output)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

# Final model
model = models.Model(inputs=[input_ids, attention_mask], outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

TypeError: 'builtins.safe_open' object is not iterable

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 6) Train
# ─────────────────────────────────────────────────────────────────────────────
ckpt = callbacks.ModelCheckpoint(filepath=os.path.join(MODEL_DIR, CHECKPOINT_FILE), save_best_only=True, monitor="val_accuracy")
es = callbacks.EarlyStopping(patience=2, restore_best_weights=True)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    callbacks=[ckpt, es]
)


NameError: name 'model' is not defined

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 7) Evaluate
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}")

NameError: name 'model' is not defined

In [10]:
# 8) Sample Predictions
# ─────────────────────────────────────────────────────────────────────────────
sample_texts = [
    "SpaceX launches new batch of Starlink satellites.",
    "The stock market crashed due to inflation fears.",
    "Manchester United wins against Liverpool in thriller match.",
    "Scientists discover a new particle at CERN."
]

sample_encodings = encode_texts(sample_texts)
sample_preds = model.predict(dict(sample_encodings))

for text, probs in zip(sample_texts, sample_preds):
    pred_class = np.argmax(probs)
    print(f"Text: {text[:50]}... Predicted Class: {pred_class} (Conf: {probs[pred_class]:.2%})")


NameError: name 'encode_texts' is not defined